# PMO Analysis — VC Portfolio Companies
Анализ оценок стартапов по модели ПМО (Персонализированная Модель Образования)

In [23]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/all_companies_pmo.csv')
print(f'Загружено строк: {len(df)}, колонок: {len(df.columns)}')
df.head(3)

Загружено строк: 2348, колонок: 24


,id,fund,name,slug,fund_url,sectors,website,description,stage,stage_year,...,ticker_symbol,acquirer,founders,pmo_score,pmo_traj,pmo_mat,pmo_collab,pmo_game,pmo_feedback,pmo_notes
0,1,a16z-speedrun,Canto Reality,canto-reality,https://speedrun.a16z.com/companies/canto-reality,AI;Infra,https://cantoreality.com,"Canto Reality, Inc. is an exploratory deployment lab building social experie...",SR001,NaN,...,NaN,NaN,Shuby Deshpande;Aryan Deshpande,0.0,0,0,0,0,0,"Продукт не является образовательным, это лаборатория по созданию социальных ..."
1,2,a16z-speedrun,Elsewhere,elsewhere,https://speedrun.a16z.com/companies/elsewhere,Gaming;AI,https://elsewhere.zone,"We make games that feel like an adventure you go on with your friends, and t...",SR001,NaN,...,NaN,NaN,Paul Sawaya,0.0,0,0,0,0,0,"Продукт не является образовательным, это развлекательная игра, поэтому модел..."
2,3,a16z-speedrun,Ex-Human,ex-human,https://speedrun.a16z.com/companies/ex-human,AI;Enterprise SaaS;Infra,https://exh.ai/,"Ex-Human is the world’s first empathetic AI platform, trained on millions of...",SR001,NaN,...,NaN,NaN,Artem Rodichev,0.0,0,0,0,0,0,Продукт не является образовательным — это платформа для создания эмоциональн...


## 1. Базовый анализ целостности данных

In [24]:
print('=== Форма датасета ===')
print(f'Строк: {df.shape[0]}, Колонок: {df.shape[1]}')

print('\n=== Типы данных ===')
print(df.dtypes)

print('\n=== Пропуски (% от общего) ===')
missing = (df.isnull().sum() / len(df) * 100).round(1)
print(missing[missing > 0].to_string())

=== Форма датасета ===
Строк: 2348, Колонок: 24

=== Типы данных ===
id                   int64
fund                object
name                object
slug                object
fund_url            object
sectors             object
website             object
description         object
stage               object
stage_year         float64
founded_year       float64
invested_year      float64
logo_url            object
source_modified     object
ticker_symbol       object
acquirer            object
founders            object
pmo_score          float64
pmo_traj             int64
pmo_mat              int64
pmo_collab           int64
pmo_game             int64
pmo_feedback         int64
pmo_notes           object
dtype: object

=== Пропуски (% от общего) ===
sectors            40.6
website             1.9
description         7.0
stage              28.7
stage_year         95.7
founded_year       51.4
invested_year      59.8
logo_url            4.6
source_modified    82.6
ticker_symbol      98

In [25]:
pmo_cols = ['pmo_score', 'pmo_traj', 'pmo_mat', 'pmo_collab', 'pmo_game', 'pmo_feedback']

print('=== PMO-оценки: базовая статистика ===')
df[pmo_cols].describe().round(2)

=== PMO-оценки: базовая статистика ===


,pmo_score,pmo_traj,pmo_mat,pmo_collab,pmo_game,pmo_feedback
count,2348.00,2348.00,2348.00,2348.00,2348.00,2348.00
mean,1.03,1.10,1.10,1.11,0.61,1.23
std,1.77,2.08,2.12,2.22,1.50,2.35
min,-1.00,-1.00,-1.00,-1.00,-1.00,-1.00
25%,0.00,0.00,0.00,0.00,0.00,0.00
50%,0.00,0.00,0.00,0.00,0.00,0.00
75%,1.60,1.00,1.00,1.00,1.00,1.00
max,7.60,10.00,10.00,10.00,10.00,10.00


In [26]:
print('=== Компании без PMO-оценки ===')
no_score = df[df['pmo_score'].isnull()]
print(f'Без оценки: {len(no_score)} ({len(no_score)/len(df)*100:.1f}%)')

print('\n=== Распределение по фондам ===')
print(df['fund'].value_counts().to_string())

=== Компании без PMO-оценки ===
Без оценки: 0 (0.0%)

=== Распределение по фондам ===
fund
a16z             839
sequoia          408
new-schools      292
a16z-speedrun    240
y-combinator     125
reach-capital     95
owl-ventures      92
gsv-ventures      87
learn-capital     78
brighteye         51
edu-capital       41


In [27]:
all_sectors = (
    df['sectors']
    .dropna()
    .str.split(';')
    .explode()
    .str.strip()
    .loc[lambda s: s != '']
    .value_counts()
)
print(f'Уникальных секторов: {len(all_sectors)}')
all_sectors

Уникальных секторов: 68


sectors
AI                                           291
Education                                    125
Consumer                                     116
Learning                                      97
Innovative Schools                            87
                                            ... 
Future of Talent @Work                         1
Workforce                                      1
Pre-K-12                                       1
Adult Consumer Learning                        1
Early Identification,Learning Differences      1
Name: count, Length: 68, dtype: int64

## 2. Все уникальные секторы

In [28]:
EDU_KEYWORDS = [
    'education',        # Education, education, Future of education, Higher Education
    'edtech',           # Edtech
    'learning',         # Learning, Learning Solutions, Learning Differences,
                        # Enterprise & Professional Learning & Talent,
                        # Adult Consumer Learning, GenAI Math Tutoring + Learning Solutions,
                        # Early Identification + Learning Differences
    'k-12',             # K-12, Pre K-12, Pre-K-12, PK-12, PreK-12, Future of Talent in PreK-12
    'k12',              # K12, K12, Higher Ed
    'post-secondary',   # Post-Secondary, Future of Talent in Post-Secondary
    'higher ed',        # K12, Higher Ed
    'teaching reimagined',  # Teaching Reimagined
    'future of talent', # Future of Talent in PreK-12, Future of Talent in Post-Secondary,
                        # Future of Talent@Work, Future of Talent @Work
]
# Намеренно исключены 'school' (Innovative Schools), 'diverse leaders' (Diverse Leaders),
# 'racial equity' (Racial Equity) и 'edge' (EDge) — эти секторы не относим к образовательным.

def is_education(sectors_str):
    if pd.isna(sectors_str):
        return False
    s = sectors_str.lower()
    return any(kw in s for kw in EDU_KEYWORDS)

edu_mask = df['sectors'].apply(is_education)
df_edu = df[edu_mask].copy().reset_index(drop=True)

print(f'Образовательных стартапов: {len(df_edu)} из {len(df)} ({len(df_edu)/len(df)*100:.1f}%)')
df_edu[['fund', 'name', 'sectors', 'pmo_score']].head(10)

Образовательных стартапов: 529 из 2348 (22.5%)


,fund,name,sectors,pmo_score
0,a16z-speedrun,Spark Space,AI;Edtech,6.4
1,a16z-speedrun,Prepp,AI;Edtech,2.0
2,a16z-speedrun,Straia,AI;Edtech;Enterprise SaaS,0.0
3,brighteye,Seven Education,Learning,1.4
4,brighteye,Ornikar,Learning,4.8
5,edu-capital,appscho,Future of education,0.0
6,edu-capital,buddy,Future of education,5.2
7,edu-capital,codary,Future of education,4.2
8,edu-capital,digischool,Future of education,2.2
9,edu-capital,opendigitaleducation,Future of education,4.4


## 3. Выделение образовательных стартапов

In [29]:
print('=== Образовательные стартапы по фондам ===')
print(df_edu['fund'].value_counts().to_string())

print('\n=== PMO-оценки образовательных стартапов ===')
print(df_edu[pmo_cols].describe().round(2))

=== Образовательные стартапы по фондам ===
fund
y-combinator     125
new-schools      114
reach-capital     95
learn-capital     62
owl-ventures      60
gsv-ventures      40
edu-capital       28
a16z-speedrun      3
brighteye          2

=== PMO-оценки образовательных стартапов ===
       pmo_score  pmo_traj  pmo_mat  pmo_collab  pmo_game  pmo_feedback
count     529.00    529.00   529.00      529.00    529.00        529.00
mean        2.91      3.03     3.23        2.75      1.80          3.72
std         1.98      2.48     2.76        2.75      2.31          3.03
min         0.00      0.00     0.00        0.00      0.00          0.00
25%         1.40      1.00     1.00        0.00      0.00          1.00
50%         2.80      2.00     2.00        2.00      1.00          3.00
75%         4.40      5.00     5.00        5.00      2.00          6.00
max         7.40     10.00    10.00       10.00     10.00         10.00


In [30]:
print('=== Топ-10 образовательных стартапов по PMO-score ===')
top10 = df_edu.sort_values('pmo_score', ascending=False)[['name', 'fund', 'sectors', 'pmo_score', 'pmo_notes']].head(10)
pd.set_option('display.max_colwidth', 80)
top10

=== Топ-10 образовательных стартапов по PMO-score ===


,name,fund,sectors,pmo_score,pmo_notes
448,Litnerd,y-combinator,Education,7.4,Litnerd сочетает геймифицированное обучение чтению и письму с живыми сообщес...
421,Alice.tech,y-combinator,Education,7.4,"Alice использует ИИ для адаптивной персонализации траектории и материалов, г..."
398,Thinkverse,reach-capital,Learning,7.2,"ИИ-репетитор по математике для школьников, предоставляющий персонализированн..."
16,knowunity,edu-capital,Future of education,7.2,Knowunity позиционируется как AI-тьютор с высокой адаптивностью материалов и...
110,Nerdy,learn-capital,education,7.2,Nerdy получает высокие оценки за персонализированную траекторию и обратную с...
294,Sizzle,owl-ventures,Pre K-12;Future of Work,7.2,Sizzle AI сочетает ИИ-персонализацию учебного контента и траектории с социал...
484,Outtalent,y-combinator,Education,7.0,Платформа подготовки к трудоустройству с сильно выраженной персональной трае...
95,Groovetime,learn-capital,education;community;gaming,7.0,Groovetime обеспечивает высокую геймификацию и мгновенную обратную связь с п...
195,New Classrooms,new-schools,Learning Solutions,6.8,Продукт предоставляет высокую персонализацию учебных траекторий и материалов...
370,Pathfinder,reach-capital,Learning,6.8,"AI-агент для семейного хоумскулинга, обеспечивающий персонализированную трае..."


## 4. Доп. анализ: согласованность классификации и pmo_score

- **Вопрос 1:** есть ли НЕ образовательные стартапы (sector не попал в `EDU_KEYWORDS`) с ненулевым `pmo_score`?
- **Вопрос 2:** есть ли среди образовательных стартапов проекты с нулевым `pmo_score`?

In [31]:
df_non_edu = df[~edu_mask].copy()

print('=== Вопрос 1: НЕ образовательные стартапы с ненулевым pmo_score ===')
print(f'Всего НЕ образовательных: {len(df_non_edu)}')
non_edu_nonzero = df_non_edu[df_non_edu['pmo_score'] != 0]
print(f'  pmo_score != 0:  {len(non_edu_nonzero)} ({len(non_edu_nonzero)/len(df_non_edu)*100:.1f}%)')
print(f'    из них > 0:    {(df_non_edu["pmo_score"] > 0).sum()}')
print(f'    из них == -1:  {(df_non_edu["pmo_score"] == -1).sum()}  (оценка не определена)')
print(f'  pmo_score == 0:  {(df_non_edu["pmo_score"] == 0).sum()}')

print('\nТоп НЕ образовательных по pmo_score:')
display(
    df_non_edu[df_non_edu['pmo_score'] > 0]
    .sort_values('pmo_score', ascending=False)[['name', 'fund', 'sectors', 'pmo_score']]
    .head(10)
)

print('\n=== Вопрос 2: образовательные стартапы с нулевым pmo_score ===')
edu_zero = df_edu[df_edu['pmo_score'] == 0]
print(f'Всего образовательных: {len(df_edu)}')
print(f'  pmo_score == 0:  {len(edu_zero)} ({len(edu_zero)/len(df_edu)*100:.1f}%)')
print(f'  pmo_score == -1: {(df_edu["pmo_score"] == -1).sum()}  (оценка не определена)')
if len(edu_zero):
    display(edu_zero[['name', 'fund', 'sectors', 'pmo_score']].head(10))

=== Вопрос 1: НЕ образовательные стартапы с ненулевым pmo_score ===
Всего НЕ образовательных: 1819
  pmo_score != 0:  346 (19.0%)
    из них > 0:    344
    из них == -1:  2  (оценка не определена)
  pmo_score == 0:  1473

Топ НЕ образовательных по pmo_score:


,name,fund,sectors,pmo_score
1150,lepaya,edu-capital,Future of work,7.6
392,CodeCombat,a16z,NaN,7.0
268,AltSchool,a16z,NaN,7.0
1636,BenchPrep,owl-ventures,Future of Work,7.0
1468,Libertas School of Memphis,new-schools,Innovative Schools,7.0
1490,Namahana Education Foundation,new-schools,Innovative Schools,7.0
1246,Toddle,gsv-ventures,NaN,7.0
1090,Tandem,brighteye,NaN,6.8
1079,Hack The Box,brighteye,Work,6.6
1509,Palm Beach School for Autism,new-schools,Innovative Schools,6.4



=== Вопрос 2: образовательные стартапы с нулевым pmo_score ===
Всего образовательных: 529
  pmo_score == 0:  86 (16.3%)
  pmo_score == -1: 0  (оценка не определена)


,name,fund,sectors,pmo_score
2,Straia,a16z-speedrun,AI;Edtech;Enterprise SaaS,0.0
5,appscho,edu-capital,Future of education,0.0
22,mentorshow,edu-capital,Future of education,0.0
29,studentpop,edu-capital,Future of education,0.0
37,Bookclub,gsv-ventures,Adult Consumer Learning,0.0
38,CampusLogic,gsv-ventures,Future of Talent in Post-Secondary,0.0
48,Fairygodboss,gsv-ventures,Future of Talent@Work,0.0
49,Glimpse,gsv-ventures,Future of Talent in PreK-12,0.0
52,Hustle,gsv-ventures,Future of Talent@Work,0.0
53,Intellispark,gsv-ventures,K-12,0.0


## 5. Экспорт стартапов

Экспортируем:
- **образовательные** проекты (сектор попадает под `EDU_KEYWORDS`);
- **НЕ образовательные** проекты с `pmo_score > 2`.

Дополнительно:
- из выгрузки **полностью исключаются** перечисленные вручную стартапы (`EXCLUDE_NAMES`);
- одна компания может встречаться под несколькими фондами — такие **дубликаты по названию** удаляем из датасета (оставляем строку с максимальным `pmo_score`).

In [32]:
export_path = '../data/edu_companies_pmo.csv'

EXCLUDE_NAMES = [
    "Neon Wild", "trophi.ai", "K12 Crypto", "Greenlight", "Medley", "Pug.AI",
    "Lablabee", "Twenix", "La Solive", "Yeschef", "Aula", "chance", "cyberguru",
    "manzalab", "simundia", "Domestika", "Ethena", "FrontRow", "Leland", "PAC",
    "Physics Wallah", "Simplilearn", "Tango", "Tract", "Nexford", "Surge Institute",
    "Authoritive", "Classplus", "Deephow", "Galena", "Gather", "Lingo Live",
    "MasterClass", "Schemata", "Thinkful", "Ubits", "Abby Care", "Meeno", "Piazza",
]
_exclude_set = {n.strip().lower() for n in EXCLUDE_NAMES}
_name_norm = df['name'].astype(str).str.strip().str.lower()
_excluded_mask = _name_norm.isin(_exclude_set)

# Исключаемые секторы: эти стартапы не должны попадать в финальный файл,
# даже если у них pmo_score > 2.
EXCLUDE_SECTORS = ["diverse leaders", "innovative schools", "racial equity", "edge"]
_exclude_sector_set = {s.strip().lower() for s in EXCLUDE_SECTORS}

def has_excluded_sector(sectors_str):
    if pd.isna(sectors_str):
        return False
    parts = {p.strip().lower() for p in str(sectors_str).split(';')}
    return bool(parts & _exclude_sector_set)

_excluded_sector_mask = df['sectors'].apply(has_excluded_sector)

# Невалидный website: пустое значение (NaN/пустая строка) или плейсхолдер "#".
_website_norm = df['website'].astype(str).str.strip()
_invalid_website_mask = df['website'].isna() | _website_norm.isin(['', '#'])

# База: образовательные ИЛИ pmo_score > 2, минус исключённые по списку имён,
# по секторам и с невалидным website.
base_mask = edu_mask | (df['pmo_score'] > 2)
export_mask = base_mask & ~_excluded_mask & ~_excluded_sector_mask & ~_invalid_website_mask
df_export = df[export_mask].copy().reset_index(drop=True)

print(f'Строк в базе (edu | pmo>2):          {int(base_mask.sum())}')
print(f'Исключено по списку EXCLUDE_NAMES:    {int((base_mask & _excluded_mask).sum())}')
print(f'Исключено по EXCLUDE_SECTORS:         {int((base_mask & ~_excluded_mask & _excluded_sector_mask).sum())}')
print(f'Исключено по пустому/"#" website:     {int((base_mask & ~_excluded_mask & ~_excluded_sector_mask & _invalid_website_mask).sum())}')
print(f'Строк до дедупликации:               {len(df_export)}')

Строк в базе (edu | pmo>2):          708
Исключено по списку EXCLUDE_NAMES:    42
Исключено по EXCLUDE_SECTORS:         91
Исключено по пустому/"#" website:     14
Строк до дедупликации:               561


In [33]:
# Удаление дубликатов: для каждой компании оставляем строку с максимальным pmo_score.
df_final = (
    df_export
    .sort_values('pmo_score', ascending=False, kind='stable')
    .drop_duplicates(subset=['name'], keep='first')
    .sort_index()
    .reset_index(drop=True)
)
df_final.to_csv(export_path, index=False)

n_edu = int(df_final['sectors'].apply(is_education).sum())
n_non_edu = len(df_final) - n_edu
print(f'Удалено дубликатов: {len(df_export) - len(df_final)}')
print(f'Экспортировано строк: {len(df_final)}, колонок: {len(df_final.columns)}')
print(f'  образовательных:       {n_edu}')
print(f'  НЕ образовательных:    {n_non_edu}')
print(f'Файл: {export_path}')

Удалено дубликатов: 43
Экспортировано строк: 518, колонок: 24
  образовательных:       472
  НЕ образовательных:    46
Файл: ../data/edu_companies_pmo.csv
